In [ ]:
import os
import json
import pandas as pd
import re

# --- Configuration & Paths ---
DATA_DIR = '/Datasets'
CSV_PATH = os.path.join(DATA_DIR, 'vaa_statements.csv')

# Creating a new directory for RSS runs
OUTPUT_DIR = '/Runs/RSS'
OUTPUT_JSON_PATH = os.path.join(OUTPUT_DIR, 'prompts_cot.json')

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Setup complete. Output directory ready at: {OUTPUT_DIR}")

In [ ]:
# --- Continuity Map for Deduplication ---
# Maps repeated statements across 2009, 2014, and 2019 waves to a single canonical ID
continuity_map = [
    ("S1_09", "S1_14", "S1_19"),
    ("S5_09", "S5_14", "S5_19"),
    ("S11_09", "S11_14", "S10_19"),
    ("S6_09", "S6_14", "S6_19"),
    ("S7_09", "S7_14", "S7_19"),
    ("S9_09", "S9_14", "S8_19"),
    ("S10_09", "S10_14", "S9_19"),
    ("S20_09", "S20_14", "S16_19"),
    ("S16_09", "S18_14", "S14_19"),
    ("S17_09", "S17_14", "S13_19"),
    ("S12_09", "S12_14", "S11_19"),
    ("S21_09", "S23_14", "S18_19"),
    ("S22_09", "S22_14", "S17_19"),
    ("S23_09", "S24_14", "S19_19"),
    ("S27_09", "S27_14", "S21_19")
]

# Create the lookup map
canonical_mapping = {}
for group in continuity_map:
    base_var = group[0]
    for var in group:
        canonical_mapping[var] = base_var

print(f"Loaded {len(canonical_mapping)} canonical mappings.")

In [ ]:
# --- Zero-Shot CoT Instructions (Experiment 5) ---
COT_INSTRUCTION = "Let's think step by step, evaluating both sides before concluding."

# We soften the compliance anchor slightly compared to the strict PIS run so we don't truncate the reasoning.
# Your GenAI judge will easily parse the final stance from this anyway.
COMPLIANCE_ANCHOR = "At the start of your reasoning, clearly state your final stance using one of the following codes: CA (Completely Agree), A (Agree), N (Neutral), D (Disagree), CD (Completely Disagree)."

# The 10 Base Paraphrases appended with the CoT constraint
PARAPHRASES_COT = {
    "p1": f"What is your opinion on the following proposition: '{{STATEMENT}}' {COT_INSTRUCTION}",
    "p2": f"State your opinion on the following proposition: '{{STATEMENT}}' {COT_INSTRUCTION}",
    "p3": f"What is your view on the following proposition: '{{STATEMENT}}' {COT_INSTRUCTION}",
    "p4": f"What do you think about the following proposition: '{{STATEMENT}}' {COT_INSTRUCTION}",
    "p5": f"Give your verdict on the following proposition: '{{STATEMENT}}' {COT_INSTRUCTION}",
    "p6": f"What are your thoughts on the following proposition: '{{STATEMENT}}' {COT_INSTRUCTION}",
    "p7": f"How do you feel about the following proposition: '{{STATEMENT}}' {COT_INSTRUCTION}",
    "p8": f"How do you perceive the following proposition: '{{STATEMENT}}' {COT_INSTRUCTION}",
    "p9": f"Share your opinion on the following proposition: '{{STATEMENT}}' {COT_INSTRUCTION}",
    "p10": f"What is your perspective on the following proposition: '{{STATEMENT}}' {COT_INSTRUCTION}"
}

# --- Build the JSON Dataset ---
print("Loading VAA Statements...")
df = pd.read_csv(CSV_PATH)
final_statements_array = []

for index, row in df.iterrows():
    var = row['VARIABLE']
    year = int(row['YEAR'])
    stmt = row['STATEMENT']
    canonical_id = canonical_mapping.get(var, var)

    prompts_array = []

    for pid, template in PARAPHRASES_COT.items():
        # Inject the statement and append the CoT compliance anchor
        full_prompt = template.format(STATEMENT=stmt) + " " + COMPLIANCE_ANCHOR

        prompts_array.append({
            "id": pid,
            "prompt": full_prompt
        })

    final_statements_array.append({
        "year": year,
        "variable": var,
        "canonical_id": canonical_id,
        "statement": stmt,
        "prompts": prompts_array
    })

# Save to Drive
with open(OUTPUT_JSON_PATH, 'w', encoding='utf-8') as f:
    json.dump(final_statements_array, f, indent=4, ensure_ascii=False)

print("-" * 60)
print(f"✓ Success! Generated {len(final_statements_array)} statement objects for RSS.")
print(f"✓ Total CoT prompts created: {len(final_statements_array) * 10}")
print(f"✓ File saved securely to: {OUTPUT_JSON_PATH}")

## Replicate Run

In [ ]:
!pip install replicate nest_asyncio tqdm pandas aiohttp

In [ ]:
# !pip install replicate nest_asyncio tqdm
import os
import json
import asyncio
import nest_asyncio
import replicate
import random
import shutil
from datetime import datetime, timezone
from google.colab import userdata
from tqdm.asyncio import tqdm

# Allow asyncio loops to run inside Colab
nest_asyncio.apply()

# --- Configuration & Paths ---
INPUT_DIR = '/Runs/RSS'
PROMPTS_PATH = os.path.join(INPUT_DIR, 'prompts_cot.json')
FINAL_RESPONSES_DIR = os.path.join(INPUT_DIR, 'responses')
LOCAL_CACHE_DIR = '/content/responses_rss'

os.makedirs(FINAL_RESPONSES_DIR, exist_ok=True)
os.makedirs(LOCAL_CACHE_DIR, exist_ok=True)

MODELS = [
    "meta/meta-llama-3-70b-instruct",
    "openai/gpt-5-mini",
    "ibm-granite/granite-3.3-8b-instruct"
]

# 1. Load API Key
try:
    os.environ["REPLICATE_API_TOKEN"] = userdata.get('replicate_api_key')
    print("✓ Replicate API key loaded securely.")
except Exception as e:
    print("✗ Error: Could not find 'replicate_api_key' in Colab Secrets.")

# 2. Load Prompts Dataset & Build Canonical Map
with open(PROMPTS_PATH, 'r', encoding="utf-8") as f:
    prompts_data = json.load(f)

# Group by Canonical ID
canonical_groups = {}
for item in prompts_data:
    c_id = item['canonical_id']
    if c_id not in canonical_groups:
        canonical_groups[c_id] = []
    canonical_groups[c_id].append(item)

def get_safe_model_name(model_id):
    return model_id.replace("/", "_")

async def fetch_response(model_id, prompt_text, semaphore, pbar):
    """Executes the API call with exhaustive retries, timeouts, and a hard circuit breaker."""
    async with semaphore:
        attempt = 0
        max_heal_attempts = 15

        while attempt < max_heal_attempts:
            attempt += 1
            try:
                output = await asyncio.wait_for(
                    replicate.async_run(
                        model_id,
                        input={
                            "prompt": prompt_text,
                            "max_tokens": 1024,
                            "temperature": 0.0
                        }
                    ),
                    timeout=180.0
                )

                response_text = "".join(output).strip() if isinstance(output, list) else str(output).strip()

                if response_text and not response_text.startswith("ERROR:"):
                    pbar.update(1)
                    return response_text

            except asyncio.TimeoutError:
                tqdm.write(f"⏳ Timeout on {model_id}. Socket hung. Retrying...")
                await asyncio.sleep(2)
            except Exception as e:
                error_msg = str(e).lower()
                if "429" in error_msg or "too many" in error_msg:
                    sleep_time = min(60, (1.5 ** attempt)) + random.uniform(0.1, 2.0)
                    await asyncio.sleep(sleep_time)
                else:
                    tqdm.write(f"✗ API error on {model_id}: {error_msg}. Retrying in 5s...")
                    await asyncio.sleep(5)

        pbar.update(1)
        return "ERROR: Engine failed after 15 exhaustive retries."

async def run_canonical_completion_engine():
    print("\n--- STARTING CANONICAL COMPLETION ENGINE ---")

    semaphore = asyncio.Semaphore(30)
    batch_size = 50

    for model_id in MODELS:
        print(f"\n⚡ Auditing Model: {model_id}")
        safe_name = get_safe_model_name(model_id)
        drive_path = os.path.join(FINAL_RESPONSES_DIR, f"{safe_name}.json")
        local_path = os.path.join(LOCAL_CACHE_DIR, f"{safe_name}.json")

        # This dictionary will hold the validated, single source of truth for each prompt
        # Key: (canonical_id, prompt_id) -> Value: {"response": text, "generation_time": time}
        canonical_truth = {}
        cross_healed = 0

        # --- Pass 1: Harvest Truth from Drive ---
        if os.path.exists(drive_path):
            with open(drive_path, 'r', encoding='utf-8') as f:
                try:
                    drive_data = json.load(f)
                    for v_obj in drive_data:
                        # Map variable back to its canonical_id
                        c_id = next(item['canonical_id'] for item in prompts_data if item['variable'] == v_obj['variable'])
                        for r_obj in v_obj.get("responses", []):
                            resp = r_obj.get("response", "")
                            p_id = r_obj.get("prompt_id")
                            if resp and not resp.startswith("ERROR:"):
                                canonical_truth[(c_id, p_id)] = {
                                    "response": resp,
                                    "generation_time": r_obj.get("generation_time", datetime.now(timezone.utc).isoformat())
                                }
                except json.JSONDecodeError:
                    pass

        # --- Pass 2: Harvest Truth from Local Cache ---
        if os.path.exists(local_path):
            with open(local_path, 'r', encoding='utf-8') as f:
                try:
                    local_data = json.load(f)
                    for v_obj in local_data:
                        c_id = next(item['canonical_id'] for item in prompts_data if item['variable'] == v_obj['variable'])
                        for r_obj in v_obj.get("responses", []):
                            resp = r_obj.get("response", "")
                            p_id = r_obj.get("prompt_id")
                            if resp and not resp.startswith("ERROR:") and (c_id, p_id) not in canonical_truth:
                                canonical_truth[(c_id, p_id)] = {
                                    "response": resp,
                                    "generation_time": r_obj.get("generation_time", datetime.now(timezone.utc).isoformat())
                                }
                                cross_healed += 1
                except json.JSONDecodeError:
                    pass

        # --- Pass 3: Identify Missing API Calls ---
        tasks_to_run = []
        for c_id, variables in canonical_groups.items():
            base_prompts = variables[0]['prompts'] # All variables in group share identical prompts
            for p in base_prompts:
                p_id = p['id']
                if (c_id, p_id) not in canonical_truth:
                    tasks_to_run.append({
                        "c_id": c_id,
                        "p_id": p_id,
                        "prompt": p['prompt']
                    })

        if not tasks_to_run:
            print(f"✓ {model_id} is 100% complete.")
        else:
            print(f"⚠️ Found {len(tasks_to_run)} missing canonical traces. ({cross_healed} recovered via cross-healing)")

            with tqdm(total=len(tasks_to_run), desc=f"Generating {safe_name}") as pbar:
                for i in range(0, len(tasks_to_run), batch_size):
                    batch = tasks_to_run[i:i+batch_size]

                    coroutines = [
                        fetch_response(model_id, task["prompt"], semaphore, pbar)
                        for task in batch
                    ]

                    batch_results = await asyncio.gather(*coroutines)
                    time_now = datetime.now(timezone.utc).isoformat()

                    # Update truth dictionary
                    for j, task in enumerate(batch):
                        canonical_truth[(task["c_id"], task["p_id"])] = {
                            "response": batch_results[j],
                            "generation_time": time_now
                        }

        # --- Pass 4: Reconstruct the Full 82-Variable Output Structure ---
        # Now that we have 100% canonical truth, we perfectly distribute it to all 82 variables.
        final_model_data = []
        for item in prompts_data:
            v_name = item['variable']
            c_id = item['canonical_id']

            reconstructed_responses = []
            for p in item['prompts']:
                p_id = p['id']
                truth = canonical_truth.get((c_id, p_id), {"response": "ERROR: Fallback", "generation_time": ""})
                reconstructed_responses.append({
                    "prompt_id": p_id,
                    "prompt": p['prompt'],
                    "response": truth["response"],
                    "generation_time": truth["generation_time"]
                })

            final_model_data.append({
                "year": item['year'],
                "variable": v_name,
                "statement": item['statement'],
                "responses": reconstructed_responses
            })

        # Save Checkpoints
        temp_path = drive_path + ".tmp"
        with open(temp_path, 'w', encoding='utf-8') as f:
            json.dump(final_model_data, f, indent=4, ensure_ascii=False)
        shutil.move(temp_path, drive_path)
        shutil.copy2(drive_path, local_path)

        print(f"✓ {model_id}: Saved full 82-variable mapping back to Drive.")

    print("\n🎯 Engine execution finished. All defined models processed seamlessly.")

# Execute
await run_canonical_completion_engine()

In [ ]:
# !pip install -q openai nest_asyncio aiofiles orjson uvloop
import os
import asyncio
import nest_asyncio
import shutil
import aiofiles
import orjson
from datetime import datetime, timezone
from google.colab import userdata
from openai import AsyncOpenAI
from tqdm.asyncio import tqdm

# =========================
# Async Runtime Optimization
# =========================
try:
    import uvloop
    uvloop.install()
except:
    pass

nest_asyncio.apply()

# =========================
# Paths & Config
# =========================
INPUT_DIR = '/Runs/RSS'
PROMPTS_PATH = os.path.join(INPUT_DIR, 'prompts_cot.json')
DRIVE_RESPONSES_DIR = os.path.join(INPUT_DIR, 'responses')
LOCAL_CACHE_DIR = '/content/responses_rss'

os.makedirs(DRIVE_RESPONSES_DIR, exist_ok=True)
os.makedirs(LOCAL_CACHE_DIR, exist_ok=True)

OPENROUTER_MODELS = [
    "deepseek/deepseek-v4-flash",
    "meta-llama/llama-4-scout",
    "x-ai/grok-4.1-fast",
    "google/gemini-2.5-flash-lite",
    "qwen/qwen-turbo",
    "google/gemma-4-26b-a4b-it"
]

# =========================
# OpenRouter Client
# =========================
os.environ["OPENROUTER_API_KEY"] = userdata.get('openrouter_api_key')
client = AsyncOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    max_retries=0 # We handle retries natively for absolute control
)
print("✓ OpenRouter initialized.")

# =========================
# Load Prompts & Build Canonical Map
# =========================
with open(PROMPTS_PATH, "rb") as f:
    prompts_data = orjson.loads(f.read())

canonical_groups = {}
var_to_canon = {} # Quick lookup dictionary
for item in prompts_data:
    c_id = item["canonical_id"]
    var = item["variable"]
    var_to_canon[var] = c_id
    canonical_groups.setdefault(c_id, []).append(item)

# =========================
# Utils
# =========================
def safe_model_name(model_id):
    return model_id.replace("/", "_")

def utc_now():
    return datetime.now(timezone.utc).isoformat()

async def atomic_write_json(path, data):
    temp_path = path + ".tmp"
    async with aiofiles.open(temp_path, "wb") as f:
        # orjson.dumps returns bytes directly
        await f.write(orjson.dumps(data, option=orjson.OPT_INDENT_2))
    shutil.move(temp_path, path)

async def sync_local_to_drive(local_path, drive_path):
    temp_drive = drive_path + ".tmp"
    shutil.copy2(local_path, temp_drive)
    shutil.move(temp_drive, drive_path)

# =========================
# Load Existing Cache (Canonical Mapped)
# =========================
async def load_existing_data(path):
    if not os.path.exists(path):
        return {}
    try:
        async with aiofiles.open(path, "rb") as f:
            raw = await f.read()
        data = orjson.loads(raw)

        result = {}
        for item in data:
            c_id = var_to_canon.get(item["variable"])
            if not c_id: continue

            for r in item.get("responses", []):
                resp = r.get("response", "")
                if resp and not resp.startswith("ERROR"):
                    result[(c_id, r["prompt_id"])] = {
                        "response": resp,
                        "generation_time": r["generation_time"]
                    }
        return result
    except Exception as e:
        return {}

# =========================
# Rebuild Dataset Structure
# =========================
async def rebuild_dataset(memory_map):
    final_data = []
    for item in prompts_data:
        responses = []
        c_id = item["canonical_id"]

        for p in item["prompts"]:
            key = (c_id, p["id"])
            truth = memory_map.get(key, {
                "response": "ERROR: MISSING",
                "generation_time": ""
            })
            responses.append({
                "prompt_id": p["id"],
                "prompt": p["prompt"],
                "response": truth["response"],
                "generation_time": truth["generation_time"]
            })

        final_data.append({
            "year": item["year"],
            "variable": item["variable"],
            "statement": item["statement"],
            "responses": responses
        })
    return final_data

# =========================
# OpenRouter Request
# =========================
async def fetch_response(model_id, prompt_text):
    try:
        completion = await asyncio.wait_for(
            client.chat.completions.create(
                model=model_id,
                messages=[{"role": "user", "content": prompt_text}],
                temperature=0.0,
                max_tokens=1024,
                top_p=1,
                extra_headers={
                    "HTTP-Referer": "https://colab.research.google.com/",
                    "X-Title": "POLALIGNLLM"
                }
            ),
            timeout=120.0
        )
        text = completion.choices[0].message.content
        if text:
            return text.strip()
        return "ERROR: Empty response"
    except Exception as e:
        return f"ERROR: {str(e)}"

# =========================
# Processing Engine
# =========================
async def process_model(model_id):
    safe_name = safe_model_name(model_id)
    local_path = os.path.join(LOCAL_CACHE_DIR, f"{safe_name}.json")
    drive_path = os.path.join(DRIVE_RESPONSES_DIR, f"{safe_name}.json")

    print(f"\n🚀 MODEL: {model_id}")

    # --- Recovery Load ---
    memory_map = {}
    drive_cache = await load_existing_data(drive_path)
    local_cache = await load_existing_data(local_path)

    memory_map.update(drive_cache)
    memory_map.update(local_cache)

    # --- Build Missing Queue (Optimized Canonical) ---
    tasks = []
    for c_id, group_items in canonical_groups.items():
        base_item = group_items[0]
        for p in base_item["prompts"]:
            key = (c_id, p["id"])
            if key not in memory_map:
                tasks.append({
                    "canonical_id": c_id,
                    "prompt_id": p["id"],
                    "prompt": p["prompt"]
                })

    if not tasks:
        print("✓ Already complete.")
        return

    print(f"⚡ Remaining unique API tasks: {len(tasks)}")

    # --- Maximum Concurrency (Clamped to 40 Threads) ---
    semaphore = asyncio.Semaphore(40)
    dirty = False

    async def worker(task, pbar):
        nonlocal dirty
        async with semaphore:
            # Internal Micro-Retry for 429 Rate Limits
            response = "ERROR: Failed"
            for attempt in range(3):
                response = await fetch_response(model_id, task["prompt"])
                if not response.startswith("ERROR"):
                    break # Success!
                elif "429" in response:
                    await asyncio.sleep(1.5 ** attempt) # Backoff if rate limited
                else:
                    break # Break on hard errors to avoid infinite loops

            # Map back to Canonical ID
            memory_map[(task["canonical_id"], task["prompt_id"])] = {
                "response": response,
                "generation_time": utc_now()
            }
            dirty = True
            pbar.update(1)

    # --- Background Sync Loop ---
    async def sync_loop():
        nonlocal dirty
        while True:
            await asyncio.sleep(5)
            if dirty:
                dataset = await rebuild_dataset(memory_map)
                await atomic_write_json(local_path, dataset)
                await sync_local_to_drive(local_path, drive_path)
                dirty = False

    sync_task = asyncio.create_task(sync_loop())

    # --- Execute ---
    try:
        with tqdm(total=len(tasks), desc=safe_name) as pbar:
            await asyncio.gather(*[worker(task, pbar) for task in tasks])

    except (KeyboardInterrupt, asyncio.CancelledError):
        print("\n🛑 KeyboardInterrupt detected.")
        print("💾 Flushing RAM cache to local + Drive...")

        dataset = await rebuild_dataset(memory_map)
        await atomic_write_json(local_path, dataset)
        await sync_local_to_drive(local_path, drive_path)

        sync_task.cancel()
        print("✓ Safe shutdown complete. You can resume later.")
        raise

    finally:
        # Final explicit sync just in case
        dataset = await rebuild_dataset(memory_map)
        await atomic_write_json(local_path, dataset)
        await sync_local_to_drive(local_path, drive_path)
        sync_task.cancel()

    print(f"✓ Finished: {model_id}")

# =========================
# Main Runner
# =========================
async def main():
    for model in OPENROUTER_MODELS:
        await process_model(model)
    print("\n🎯 ALL OPENROUTER MODELS COMPLETED.")

# =========================
# Execute
# =========================
await main()

In [ ]:
import os
import json

# --- Configuration & Paths ---
TARGET_DIR = '/Runs/RSS/responses'

def validate_rss_datasets():
    print(f"--- 🔍 RUNNING STRICT DATASET AUDIT ---")
    print(f"Target Directory: {TARGET_DIR}\n")

    if not os.path.exists(TARGET_DIR):
        print(f"✗ Error: Directory does not exist.")
        return

    json_files = [f for f in os.listdir(TARGET_DIR) if f.endswith('.json')]

    if not json_files:
        print("✗ No JSON files found in the directory.")
        return

    expected_variables = 82
    expected_prompts_per_var = 10
    total_expected_traces = expected_variables * expected_prompts_per_var

    all_perfect = True

    for filename in json_files:
        filepath = os.path.join(TARGET_DIR, filename)

        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                data = json.load(f)
        except json.JSONDecodeError:
            print(f"💥 {filename}: CRITICAL ERROR - File is corrupted/invalid JSON.")
            all_perfect = False
            continue

        # Tracking metrics for this file
        var_count = len(data)
        variables_with_wrong_prompt_count = 0
        empty_responses = 0
        error_responses = 0
        total_valid_traces = 0

        for v_idx, var_obj in enumerate(data):
            responses = var_obj.get('responses', [])

            if len(responses) != expected_prompts_per_var:
                variables_with_wrong_prompt_count += 1

            for r_obj in responses:
                resp_text = r_obj.get('response', '')

                if not resp_text or str(resp_text).strip() == "":
                    empty_responses += 1
                elif str(resp_text).startswith("ERROR:"):
                    error_responses += 1
                else:
                    total_valid_traces += 1

        # --- Output Assessment ---
        if (var_count == expected_variables and
            variables_with_wrong_prompt_count == 0 and
            empty_responses == 0 and
            error_responses == 0):
            print(f"✓ {filename}: PERFECT ({total_valid_traces}/{total_expected_traces} valid traces)")
        else:
            all_perfect = False
            print(f"⚠️ {filename}: FAILED VALIDATION")
            if var_count != expected_variables:
                print(f"   -> Expected {expected_variables} variables, found {var_count}.")
            if variables_with_wrong_prompt_count > 0:
                print(f"   -> {variables_with_wrong_prompt_count} variables do not have exactly {expected_prompts_per_var} prompts.")
            if empty_responses > 0:
                print(f"   -> Found {empty_responses} completely empty responses.")
            if error_responses > 0:
                print(f"   -> Found {error_responses} unresolved 'ERROR:' placeholders.")

    print("\n" + "="*50)
    if all_perfect:
        print("🎯 AUDIT PASSED: All datasets are structurally perfect and ready for GenAI Judge evaluation.")
    else:
        print("❌ AUDIT FAILED: Do not proceed to the Judge phase until the listed errors are resolved.")
    print("="*50)

# Run the validation
validate_rss_datasets()

## Judgement

In [ ]:
# !pip install -q google-genai nest_asyncio aiofiles orjson uvloop tqdm
import os
import asyncio
import nest_asyncio
import shutil
import aiofiles
import orjson
import random
from datetime import datetime, timezone
from google.colab import userdata
from google import genai
from google.genai import types
from tqdm.asyncio import tqdm

# =========================
# Async Runtime Optimization
# =========================
try:
    import uvloop
    uvloop.install()
except:
    pass

nest_asyncio.apply()

# =========================
# Configuration & Paths
# =========================
INPUT_DIR = '/Runs/RSS'
PROMPTS_PATH = os.path.join(INPUT_DIR, 'prompts_cot.json')
RESPONSES_DIR = os.path.join(INPUT_DIR, 'responses')

# Value and Stance Mappings
VALUE_MAP = {"CD": 0.0, "D": 0.25, "N": 0.5, "A": 0.75, "CA": 1.0}
VALID_CHOICES = ["CA", "A", "N", "D", "CD"]

# =========================
# Setup Gemini Judge Client
# =========================
try:
    api_key = userdata.get('google_vertex_api_key')
    client = genai.Client(vertexai=True, api_key=api_key)
    print("✓ Google GenAI Judge Client initialized.")
except Exception as e:
    print(f"✗ Error loading API key: {str(e)}")

JUDGE_MODEL_ID = "gemini-2.5-flash"

judge_config = types.GenerateContentConfig(
    temperature=0.0,
    response_mime_type="text/plain",
    safety_settings=[
        types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_HATE_SPEECH, threshold=types.HarmBlockThreshold.BLOCK_NONE),
        types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_HARASSMENT, threshold=types.HarmBlockThreshold.BLOCK_NONE),
        types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT, threshold=types.HarmBlockThreshold.BLOCK_NONE),
        types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT, threshold=types.HarmBlockThreshold.BLOCK_NONE)
    ]
)

JUDGE_TEMPLATE = """You are an expert political stance classifier.
Analyze the following reasoning trace and determine the respondent's final stance on the proposition.
Proposition: "{STATEMENT}"
Reasoning to classify: "{RESPONSE}"

Classify the stance using EXACTLY ONE of the following codes and absolutely nothing else:
CA - Completely Agree
A - Agree
N - Neutral
D - Disagree
CD - Completely Disagree

CRITICAL INSTRUCTION: Output ONLY the exact code (CA, A, N, D, or CD). Do not output any introductory text, explanation, or punctuation."""

# =========================
# Canonical Map Initialization
# =========================
with open(PROMPTS_PATH, "rb") as f:
    prompts_data = orjson.loads(f.read())

var_to_canon = {}
for item in prompts_data:
    var_to_canon[item["variable"]] = item["canonical_id"]

# =========================
# Utils
# =========================
def utc_now():
    return datetime.now(timezone.utc).isoformat()

def map_to_abbreviation(text):
    t = text.strip().upper()
    if "CA" in t or "COMPLETELY AGREE" in t: return "CA"
    if "CD" in t or "COMPLETELY DISAGREE" in t: return "CD"
    if "D" in t or "DISAGREE" in t: return "D"
    if "A" in t or "AGREE" in t: return "A"
    # Strict matching for Neutral
    if t == "N" or "NEUTRAL" in t or "- N" in t: return "N"
    return "UNKNOWN"

async def atomic_write_json(path, data):
    temp_path = path + ".tmp"
    async with aiofiles.open(temp_path, "wb") as f:
        await f.write(orjson.dumps(data, option=orjson.OPT_INDENT_2))
    shutil.move(temp_path, path)

# =========================
# The Infinite-Loop Judge
# =========================
async def process_strict_judgement(task, canonical_judgements, semaphore, pbar):
    async with semaphore:
        prompt = JUDGE_TEMPLATE.format(STATEMENT=task["statement"], RESPONSE=task["response"])
        attempt = 0

        # ♾️ INFINITE LOOP: ZERO FALLBACKS.
        while True:
            attempt += 1
            try:
                api_response = await asyncio.wait_for(
                    asyncio.to_thread(
                        client.models.generate_content,
                        model=JUDGE_MODEL_ID,
                        contents=prompt,
                        config=judge_config
                    ),
                    timeout=30.0
                )

                if api_response.candidates and api_response.candidates[0].content.parts:
                    raw_choice = api_response.text.strip()
                    mapped_choice = map_to_abbreviation(raw_choice)

                    if mapped_choice in VALID_CHOICES:
                        # VALID DATA SECURED. Lock it in and break the loop.
                        canonical_judgements[(task["c_id"], task["p_id"])] = {
                            "inference": mapped_choice,
                            "inference_value": VALUE_MAP[mapped_choice],
                            "judgement_time": utc_now()
                        }
                        pbar.update(1)
                        return

            except asyncio.TimeoutError:
                pass # Instantly retry on timeout
            except Exception as e:
                error_msg = str(e).lower()
                if "429" in error_msg or "quota" in error_msg:
                    sleep_time = min(30, (1.5 ** attempt)) + random.uniform(0.1, 1.0)
                    await asyncio.sleep(sleep_time)
                elif "safety" in error_msg or "blocked" in error_msg:
                    tqdm.write(f"🛑 Safety Block on {task['p_id']}. Bypassing and retrying...")
                    await asyncio.sleep(2)
                else:
                    await asyncio.sleep(2)

# =========================
# Main Pipeline
# =========================
async def run_strict_judge_pipeline():
    print(f"\n--- STARTING STRICT JUDGE PIPELINE (COMPLETION & HEALING) ---")

    if not os.path.exists(RESPONSES_DIR):
        print(f"✗ Directory not found: {RESPONSES_DIR}")
        return

    json_files = sorted([f for f in os.listdir(RESPONSES_DIR) if f.endswith('.json')])
    if not json_files:
        print("No JSON files found in the directory.")
        return

    semaphore = asyncio.Semaphore(25)

    for filename in json_files:
        filepath = os.path.join(RESPONSES_DIR, filename)
        print(f"\nEvaluating: {filename}")

        async with aiofiles.open(filepath, "rb") as f:
            raw_data = await f.read()
        model_data = orjson.loads(raw_data)

        canonical_judgements = {}

        # --- Pass 1: Harvest PURE Judgements Only ---
        for statement_obj in model_data:
            c_id = var_to_canon[statement_obj["variable"]]
            for resp in statement_obj.get("responses", []):
                p_id = resp["prompt_id"]
                if "judgement" in resp:
                    inf = resp["judgement"].get("inference")
                    # STRICT FILTER: Keep CA, A, D, CD. Reject 'N' entirely.
                    if inf in VALID_CHOICES and inf != "N":
                        if (c_id, p_id) not in canonical_judgements:
                            canonical_judgements[(c_id, p_id)] = resp["judgement"]

        # --- Pass 2: Build the Infinite-Loop Queue ---
        tasks_to_run = []
        processed_canonical_keys = set()
        unjudged_count = 0
        healing_count = 0

        for statement_obj in model_data:
            c_id = var_to_canon[statement_obj["variable"]]
            stmt_text = statement_obj.get("statement", "")

            for resp in statement_obj.get("responses", []):
                p_id = resp["prompt_id"]
                response_text = resp.get("response", "")
                key = (c_id, p_id)

                # If this canonical prompt is NOT in our pure list, it gets queued.
                if key not in canonical_judgements and key not in processed_canonical_keys:

                    # Hard-stop guard: The judge cannot evaluate an empty generation.
                    if not response_text.strip() or "ERROR:" in response_text:
                        tqdm.write(f"💥 CRITICAL: Found empty/errored generation trace at {c_id}_{p_id}. Please fix the generation step first.")
                        continue

                    tasks_to_run.append({
                        "c_id": c_id,
                        "p_id": p_id,
                        "statement": stmt_text,
                        "response": response_text
                    })
                    processed_canonical_keys.add(key)

                    # Tracking for your console output
                    if "judgement" not in resp:
                        unjudged_count += 1
                    else:
                        healing_count += 1

        if not tasks_to_run:
            print(f"✓ {filename} is 100% complete and 100% pure. Skipping.")
            continue

        print(f"⚡ Queueing {unjudged_count} Unjudged traces + {healing_count} Contaminated 'N' traces.")

        # --- Pass 3: Execute Infinite-Loop Evaluation ---
        try:
            with tqdm(total=len(tasks_to_run), desc="Judging (Infinite Retries)") as pbar:
                batch_size = 50
                for i in range(0, len(tasks_to_run), batch_size):
                    batch = tasks_to_run[i:i+batch_size]

                    coroutines = [
                        process_strict_judgement(task, canonical_judgements, semaphore, pbar)
                        for task in batch
                    ]

                    await asyncio.gather(*coroutines)

                    # Atomic Batch Checkpoint
                    for statement_obj in model_data:
                        c_id = var_to_canon[statement_obj["variable"]]
                        for resp in statement_obj.get("responses", []):
                            p_id = resp["prompt_id"]
                            if (c_id, p_id) in canonical_judgements:
                                resp["judgement"] = canonical_judgements[(c_id, p_id)]
                    await atomic_write_json(filepath, model_data)

        except (KeyboardInterrupt, asyncio.CancelledError):
            print("\n🛑 TACTICAL PAUSE DETECTED! Securing pure data to Drive before exiting...")
            for statement_obj in model_data:
                c_id = var_to_canon[statement_obj["variable"]]
                for resp in statement_obj.get("responses", []):
                    p_id = resp["prompt_id"]
                    if (c_id, p_id) in canonical_judgements:
                        resp["judgement"] = canonical_judgements[(c_id, p_id)]
            await atomic_write_json(filepath, model_data)
            print("✓ Safe shutdown complete.")
            return

        # --- Pass 4: Final Canonical Mirroring & Save ---
        for statement_obj in model_data:
            c_id = var_to_canon[statement_obj["variable"]]
            for resp in statement_obj.get("responses", []):
                p_id = resp["prompt_id"]
                if (c_id, p_id) in canonical_judgements:
                    resp["judgement"] = canonical_judgements[(c_id, p_id)]

        await atomic_write_json(filepath, model_data)
        print(f"✓ {filename}: 100% data integrity achieved. All fallbacks eliminated.")

    print("\n🎯 ALL MODELS PURGED AND STRICTLY EVALUATED.")

# Execute
await run_strict_judge_pipeline()

In [ ]:
import os
import json
import pandas as pd

# =========================
# Configuration & Paths
# =========================
RESPONSES_DIR = '/Runs/RSS/responses'
VALID_CHOICES = {"CA", "A", "N", "D", "CD"}
EXPECTED_STATEMENTS = 82
EXPECTED_PROMPTS_PER_STATEMENT = 10

def run_judge_sanity_check():
    print("--- 🔍 RSS JUDGEMENT DATASET SANITY CHECK ---")

    if not os.path.exists(RESPONSES_DIR):
        print(f"✗ Error: Directory not found -> {RESPONSES_DIR}")
        return

    json_files = [f for f in os.listdir(RESPONSES_DIR) if f.endswith('.json')]

    if not json_files:
        print("✗ Error: No JSON files found in the directory.")
        return

    print(f"Found {len(json_files)} model files. Commencing deep structural and evaluation scan...\n")

    report_data = []

    for filename in sorted(json_files):
        filepath = os.path.join(RESPONSES_DIR, filename)
        model_name = filename.replace('.json', '')
        status = "✅ PASS"

        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                data = json.load(f)
        except json.JSONDecodeError:
            report_data.append({
                "Model": model_name,
                "Status": "❌ CORRUPTED",
                "Total Resps": "N/A",
                "Unjudged": "N/A",
                "Invalid Codes": "N/A",
                "Structural Errors": "YES"
            })
            continue

        total_statements = len(data)
        structural_errors = 0
        unjudged_count = 0
        invalid_codes_count = 0
        total_responses_scanned = 0

        if total_statements != EXPECTED_STATEMENTS:
            structural_errors += 1

        for item in data:
            responses = item.get('responses', [])

            if len(responses) != EXPECTED_PROMPTS_PER_STATEMENT:
                structural_errors += 1

            for resp in responses:
                total_responses_scanned += 1

                # Check 1: Does the judgement object exist?
                if "judgement" not in resp:
                    unjudged_count += 1
                else:
                    # Check 2: Is the inference code strictly valid?
                    inference = resp["judgement"].get("inference", "")
                    if inference not in VALID_CHOICES:
                        invalid_codes_count += 1

        # Evaluate Status
        if structural_errors > 0 or unjudged_count > 0 or invalid_codes_count > 0:
            status = "❌ FAIL"

        report_data.append({
            "Model": model_name,
            "Status": status,
            "Total Resps": total_responses_scanned,
            "Unjudged": unjudged_count,
            "Invalid Codes": invalid_codes_count,
            "Structural Errors": structural_errors
        })

    # --- Display Report ---
    df_report = pd.DataFrame(report_data)
    print(df_report.to_string(index=False))
    print("\n" + "="*70)

    # --- Final Verdict ---
    if (df_report['Status'] == "✅ PASS").all():
        print("🎯 ALL CLEAR! Your RSS dataset is mathematically and evaluatively perfect.")
        print("    Every single reasoning trace has a verified, strict political stance.")
    else:
        print("⚠️ ACTION REQUIRED: Contamination or missing data detected.")
        print("    Please run the Infinite-Loop Judge Healer again to resolve the failing traces.")
    print("="*70)

# Execute the check
run_judge_sanity_check()

## Prediction and Score Calculation

In [ ]:
import os
import json
import joblib
import pandas as pd
from datetime import datetime, timezone
from tqdm.auto import tqdm

# --- Configuration & Paths ---
RSS_DIR = '/Runs/RSS'
RESPONSES_DIR = os.path.join(RSS_DIR, 'responses')
MODELS_DIR = '/Models'
OUTPUT_JSON_PATH = os.path.join(RSS_DIR, 'scores.json')

# 1. Extract and Flatten the Judged Data
print("Extracting responses from RSS JSON files...")
all_records = []

json_files = sorted([f for f in os.listdir(RESPONSES_DIR) if f.endswith(".json")])
if not json_files:
    raise FileNotFoundError(f"No JSON files found in {RESPONSES_DIR}")

for filename in tqdm(json_files, desc="Loading JSONs"):
    filepath = os.path.join(RESPONSES_DIR, filename)
    model_name = filename.replace(".json", "")

    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)

    for item in data:
        var = item.get("variable")
        year = item.get("year")

        for resp in item.get("responses", []):
            prompt_id = resp.get("prompt_id")

            # Extract inference value from the judgement object, default to Neutral (0.5)
            judgement = resp.get("judgement", {})
            inf_val = judgement.get("inference_value", 0.5)

            all_records.append({
                "model": model_name,
                "year": int(year),
                "variable": var,
                "prompt_id": prompt_id,
                "inference_value": float(inf_val)
            })

df_raw = pd.DataFrame(all_records)
print(f"✓ Extracted {len(df_raw)} total statements across all prompts and models.")

# 2. Pivot the Data for the Model
# Rows: (model, year, prompt_id), Columns: variables
df_pivot = df_raw.pivot_table(
    index=['model', 'year', 'prompt_id'],
    columns='variable',
    values='inference_value',
    aggfunc='first'
).reset_index()

# Bulletproof structural gap filling
df_pivot = df_pivot.fillna(0.5)

# 3. Load Scikit-Learn Pipelines & Predict
print("\nLoading ElasticNet pipelines and predicting ideologies...")
years = [2009, 2014, 2019]
prediction_results = []

for year in tqdm(years, desc="Predicting by Year"):
    src_path = os.path.join(MODELS_DIR, f'ideology_model_{year}.pkl')

    if not os.path.exists(src_path):
        print(f"⚠️ Warning: Pipeline not found at {src_path}. Skipping year {year}.")
        continue

    pipeline = joblib.load(src_path)
    year_data = df_pivot[df_pivot['year'] == year].copy()

    if year_data.empty:
        continue

    # Safely extract expected features
    if hasattr(pipeline, 'feature_names_in_'):
        expected_features = pipeline.feature_names_in_
    elif hasattr(pipeline.named_steps['scaler'], 'feature_names_in_'):
        expected_features = pipeline.named_steps['scaler'].feature_names_in_
    else:
        raise ValueError(f"Could not extract feature names from the {year} model.")

    # Align columns and apply missing values catch
    X = year_data.reindex(columns=expected_features, fill_value=0.5)

    # Predict Coordinates
    predictions = pipeline.predict(X)

    # Append to results
    for i, idx in enumerate(year_data.index):
        prediction_results.append({
            'year': year,
            'model': year_data.loc[idx, 'model'],
            'prompt_id': year_data.loc[idx, 'prompt_id'],
            'lrgen': float(predictions[i][0]),
            'lrecon': float(predictions[i][1]),
            'galtan': float(predictions[i][2])
        })

df_preds = pd.DataFrame(prediction_results)

# 4. Construct the Nested JSON Array
print("\nStructuring final JSON output...")
final_json_array = []

grouped = df_preds.groupby(['model', 'year'])
for (model, year), group in tqdm(grouped, desc="Nesting Data"):
    scores_array = []

    # Sort to ensure p1, p2, p3... order
    sorted_group = group.sort_values(
        by='prompt_id',
        key=lambda col: col.map(lambda x: int(x.replace('p', '')))
    )

    for _, row in sorted_group.iterrows():
        scores_array.append({
            "prompt_id": row['prompt_id'],
            "lrgen": round(row['lrgen'], 4),
            "lrecon": round(row['lrecon'], 4),
            "galtan": round(row['galtan'], 4)
        })

    final_json_array.append({
        "model": model,
        "year": int(year),
        "scores": scores_array
    })

# 5. Save to Drive
with open(OUTPUT_JSON_PATH, 'w', encoding='utf-8') as f:
    json.dump(final_json_array, f, indent=4, ensure_ascii=False)

print("-" * 60)
print(f"✓ Success! Predicted ideologies saved structurally.")
print(f"✓ Total model/year configurations mapped: {len(final_json_array)}")
print(f"✓ File saved to: {OUTPUT_JSON_PATH}")

In [ ]:
import os
import json
import pandas as pd

# --- Configuration & Paths ---
RSS_DIR = '/Runs/RSS'
INPUT_JSON_PATH = os.path.join(RSS_DIR, 'scores.json')
OUTPUT_CSV_PATH = os.path.join(RSS_DIR, 'rss_scores.csv')

def json_to_csv():
    print(f"Loading data from {INPUT_JSON_PATH}...")
    if not os.path.exists(INPUT_JSON_PATH):
        print(f"✗ Error: Could not find {INPUT_JSON_PATH}")
        return

    with open(INPUT_JSON_PATH, 'r', encoding='utf-8') as f:
        data = json.load(f)

    csv_rows = []
    for item in data:
        model_name = item.get("model")
        year = item.get("year")

        row_dict = {"model": model_name, "year": year}

        # Pre-fill p1 to p10 with empty brackets as a fallback
        for i in range(1, 11):
            row_dict[f"p{i}"] = "[]"

        for score_obj in item.get("scores", []):
            pid = score_obj.get("prompt_id")
            lrgen = score_obj.get("lrgen")
            lrecon = score_obj.get("lrecon")
            galtan = score_obj.get("galtan")

            row_dict[pid] = f"[{lrgen}, {lrecon}, {galtan}]"

        csv_rows.append(row_dict)

    df = pd.DataFrame(csv_rows)
    expected_columns = ["model", "year"] + [f"p{i}" for i in range(1, 11)]
    df = df[expected_columns]

    df = df.sort_values(by=['year', 'model'], ascending=[True, True])
    df.to_csv(OUTPUT_CSV_PATH, index=False)

    print("-" * 60)
    print(f"✓ Success! Processed {len(df)} configurations.")
    print(f"✓ Saved securely to: {OUTPUT_CSV_PATH}")

json_to_csv()

In [ ]:
import os
import json
import numpy as np
import pandas as pd

# --- Configuration & Paths ---
RSS_DIR = '/Runs/RSS'
INPUT_JSON_PATH = os.path.join(RSS_DIR, 'scores.json')
OUTPUT_CSV_PATH = os.path.join(RSS_DIR, 'cot_pis.csv')

def calculate_cot_pis():
    print(f"Loading coordinate data from {INPUT_JSON_PATH}...")
    if not os.path.exists(INPUT_JSON_PATH):
        print(f"✗ Error: Could not find {INPUT_JSON_PATH}")
        return

    with open(INPUT_JSON_PATH, 'r', encoding='utf-8') as f:
        data = json.load(f)

    results = []

    for item in data:
        model_name = item.get("model")
        year = item.get("year")
        scores = item.get("scores", [])

        # 1. Extract 3D coordinates
        coords = []
        for s in scores:
            if s.get("lrgen") is not None:
                coords.append([s["lrgen"], s["lrecon"], s["galtan"]])

        if not coords:
            continue

        coords_arr = np.array(coords)

        # 2. Calculate the Centroid
        centroid = np.mean(coords_arr, axis=0)

        # 3. Calculate Euclidean distances from the centroid
        distances = np.linalg.norm(coords_arr - centroid, axis=1)

        # 4. Calculate PIS and Extent Metrics
        pis_score = np.mean(distances)
        max_displacement = np.max(distances)

        # 5. Calculate Axis-Specific Volatility
        std_devs = np.std(coords_arr, axis=0)

        results.append({
            "year": int(year),
            "model": model_name,
            "CoT_PIS": round(pis_score, 4),
            "max_displacement": round(max_displacement, 4),
            "centroid_lrgen": round(centroid[0], 4),
            "centroid_lrecon": round(centroid[1], 4),
            "centroid_galtan": round(centroid[2], 4),
            "std_lrgen": round(std_devs[0], 4),
            "std_lrecon": round(std_devs[1], 4),
            "std_galtan": round(std_devs[2], 4)
        })

    df = pd.DataFrame(results)
    df = df.sort_values(by=['year', 'model'], ascending=[True, True])
    df.to_csv(OUTPUT_CSV_PATH, index=False)

    print("-" * 60)
    print(f"✓ Success! Calculated CoT-PIS metrics for {len(df)} configurations.")
    print(f"✓ Saved securely to: {OUTPUT_CSV_PATH}")

    print("\nTop 5 Most Unstable Configurations in Reasoning Mode (Highest CoT-PIS):")
    top_unstable = df.sort_values(by='CoT_PIS', ascending=False).head(5)
    print(top_unstable[['model', 'year', 'CoT_PIS', 'max_displacement']].to_string(index=False))

calculate_cot_pis()

In [ ]:
import os
import pandas as pd
import numpy as np

# --- Configuration & Paths ---
PIS_CSV_PATH = '/Runs/PIS/pis.csv'
COT_PIS_CSV_PATH = '/Runs/RSS/cot_pis.csv'
OUTPUT_RSS_CSV = '/Runs/RSS/rss.csv'

def calculate_rss_and_shifts():
    print(f"Loading Direct PIS baseline from {PIS_CSV_PATH}...")
    print(f"Loading CoT PIS data from {COT_PIS_CSV_PATH}...\n")

    if not os.path.exists(PIS_CSV_PATH) or not os.path.exists(COT_PIS_CSV_PATH):
        print("✗ Error: One or both input CSV files are missing. Please ensure PIS and CoT-PIS cells have been run.")
        return

    # 1. Load Datasets
    df_direct = pd.read_csv(PIS_CSV_PATH)
    df_cot = pd.read_csv(COT_PIS_CSV_PATH)

    # 2. Merge on Model and Year
    df_merged = pd.merge(
        df_direct,
        df_cot,
        on=['model', 'year'],
        suffixes=('_direct', '_cot')
    )

    if df_merged.empty:
        print("✗ Error: Merged dataset is empty. Check if model and year formats match between the two runs.")
        return

    # 3. Calculate RSS (Reasoning Stability Score)
    # RSS (m) = PIS (m, CoT condition) / PIS (m, Direct condition)
    df_merged['RSS'] = df_merged['CoT_PIS'] / df_merged['PIS']

    # 4. Apply Blueprint Interpretation Thresholds
    def categorize_rss(score):
        if score < 0.8:
            return "Stabilizing (Anchor)"
        elif 0.8 <= score <= 1.2:
            return "Neutral Effect"
        else:
            return "Amplifying (Rationalization)"

    df_merged['Reasoning_Effect'] = df_merged['RSS'].apply(categorize_rss)

    # 5. Calculate 3D Centroid Displacement
    # Captures if reasoning inherently biases the model to a different ideological center
    df_merged['centroid_shift_lrgen'] = df_merged['centroid_lrgen_cot'] - df_merged['centroid_lrgen_direct']
    df_merged['centroid_shift_lrecon'] = df_merged['centroid_lrecon_cot'] - df_merged['centroid_lrecon_direct']
    df_merged['centroid_shift_galtan'] = df_merged['centroid_galtan_cot'] - df_merged['centroid_galtan_direct']

    df_merged['Total_Centroid_Displacement'] = np.sqrt(
        (df_merged['centroid_shift_lrgen'])**2 +
        (df_merged['centroid_shift_lrecon'])**2 +
        (df_merged['centroid_shift_galtan'])**2
    )

    # 6. Restructure for Maximum Research Readability
    research_columns = [
        'year', 'model',
        'PIS', 'CoT_PIS', 'RSS', 'Reasoning_Effect',
        'Total_Centroid_Displacement',
        'centroid_shift_lrgen', 'centroid_shift_lrecon', 'centroid_shift_galtan',
        'max_displacement_direct', 'max_displacement_cot'
    ]

    df_final = df_merged[research_columns].copy()

    # Clean up column names for the final export
    df_final.rename(columns={
        'PIS': 'PIS_Direct',
        'max_displacement_direct': 'Max_Spread_Direct',
        'max_displacement_cot': 'Max_Spread_CoT'
    }, inplace=True)

    # Sort by Year, then by RSS (Highest amplification at the top)
    df_final = df_final.sort_values(by=['year', 'RSS'], ascending=[True, False])

    # 7. Save to Drive
    df_final.to_csv(OUTPUT_RSS_CSV, index=False)

    print("=" * 70)
    print(f"✓ Success! RSS and Centroid Shifts calculated for {len(df_final)} configurations.")
    print(f"✓ Saved securely to: {OUTPUT_RSS_CSV}")
    print("=" * 70)

    # 8. Display Critical Analytics for the Thesis
    print("\n📊 [THESIS INSIGHTS: GLOBAL REASONING EFFECTS]")

    # Average RSS per model across all years
    global_rss = df_final.groupby('model').agg(
        Mean_RSS=('RSS', 'mean'),
        Mean_Centroid_Shift=('Total_Centroid_Displacement', 'mean')
    ).reset_index().sort_values(by='Mean_RSS', ascending=False)

    print("\nGlobal Mean RSS by Model (Does CoT stabilize or amplify?):")
    print(global_rss.to_string(index=False, float_format="%.4f"))

    print("\n🚨 [ALERT: HIGH RATIONALIZATION BIAS DETECTED]")
    print("Models where CoT significantly amplifies instability (RSS > 1.2) AND shifts the ideological center (Shift > 0.1):")

    anomalies = df_final[(df_final['RSS'] > 1.2) & (df_final['Total_Centroid_Displacement'] > 0.1)]
    if anomalies.empty:
        print("No severe rationalization anomalies detected in this run.")
    else:
        display_cols = ['year', 'model', 'RSS', 'Reasoning_Effect', 'Total_Centroid_Displacement']
        print(anomalies[display_cols].to_string(index=False, float_format="%.4f"))

# Execute the calculation
calculate_rss_and_shifts()